# Customer Engagement & Product Utilization Analytics
## Phase 2: Data Preprocessing & Exploratory Data Analysis

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

### 1. Data Ingestion & Validation

In [2]:
# Load dataset
df = pd.read_csv('European_Bank.csv')

print("=== Dataset Info ===")
df.info()
print("\n=== Missing Values ===")
print(df.isnull().sum())
print("\n=== Head ===")
display(df.head())
print("\n=== Summary Stats ===")
display(df.describe())

=== Dataset Info ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Year             10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  object 
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  object 
 5   Gender           10000 non-null  object 
 6   Age              10000 non-null  int64  
 7   Tenure           10000 non-null  int64  
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(2), int64(9), object(3)
memory usage: 1.1+ MB

=== Missing Values ===
Year               0
CustomerId         0
Surname

,Year,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,2025,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2025,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,2025,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,2025,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,2025,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0



=== Summary Stats ===


,Year,CustomerId,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
count,10000.0,1.000000e+04,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.00000,10000.000000,10000.000000,10000.000000
mean,2025.0,1.569094e+07,650.528800,38.921800,5.012800,76485.889288,1.530200,0.70550,0.515100,100090.239881,0.203700
std,0.0,7.193619e+04,96.653299,10.487806,2.892174,62397.405202,0.581654,0.45584,0.499797,57510.492818,0.402769
min,2025.0,1.556570e+07,350.000000,18.000000,0.000000,0.000000,1.000000,0.00000,0.000000,11.580000,0.000000
25%,2025.0,1.562853e+07,584.000000,32.000000,3.000000,0.000000,1.000000,0.00000,0.000000,51002.110000,0.000000
50%,2025.0,1.569074e+07,652.000000,37.000000,5.000000,97198.540000,1.000000,1.00000,1.000000,100193.915000,0.000000
75%,2025.0,1.575323e+07,718.000000,44.000000,7.000000,127644.240000,2.000000,1.00000,1.000000,149388.247500,0.000000
max,2025.0,1.581569e+07,850.000000,92.000000,10.000000,250898.090000,4.000000,1.00000,1.000000,199992.480000,1.000000


### 2. Engagement Classification
Creating engagement profiles:
- Active engaged customers
- Inactive disengaged customers
- Active but low-product customers
- Inactive high-balance customers

In [3]:
non_zero_balance_median = df[df['Balance'] > 0]['Balance'].median()
df['Profile'] = 'Other'
df.loc[(df['IsActiveMember'] == 1) & (df['NumOfProducts'] > 1), 'Profile'] = 'Active Engaged'
df.loc[(df['IsActiveMember'] == 0) & (df['NumOfProducts'] == 1), 'Profile'] = 'Inactive Disengaged'
df.loc[(df['IsActiveMember'] == 1) & (df['NumOfProducts'] <= 1), 'Profile'] = 'Active Low-Product'
df.loc[(df['IsActiveMember'] == 0) & (df['Balance'] >= non_zero_balance_median), 'Profile'] = 'Inactive High-Balance'

print("=== Profile Distribution ===")
display(df['Profile'].value_counts().to_frame())

print("\n=== Profile Churn Rate ===")
display(df.groupby('Profile')['Exited'].mean().to_frame().sort_values(by='Exited', ascending=False))

=== Profile Distribution ===


,count
Profile,
Active Engaged,2588
Active Low-Product,2563
Other,1813
Inactive High-Balance,1569
Inactive Disengaged,1467



=== Profile Churn Rate ===


,Exited
Profile,
Inactive Disengaged,0.383776
Inactive High-Balance,0.312301
Active Low-Product,0.189231
Other,0.137341
Active Engaged,0.096600


### 3. Product Utilization Analysis

In [4]:
print("=== Churn by NumOfProducts ===")
display(df.groupby('NumOfProducts')['Exited'].agg(['count', 'mean']))

=== Churn by NumOfProducts ===


,count,mean
NumOfProducts,,
1,5084,0.277144
2,4590,0.075817
3,266,0.827068
4,60,1.000000


### 4. Financial Commitment vs Engagement Analysis

In [5]:
# Salary-balance mismatch (high salary, low balance)
high_salary_threshold = df['EstimatedSalary'].median()
low_balance_threshold = 0 # Mostly zero
    
df['Salary_Balance_Mismatch'] = np.where((df['EstimatedSalary'] > high_salary_threshold) & (df['Balance'] == 0), 1, 0)
print("=== Salary-Balance Mismatch Churn ===")
display(df.groupby('Salary_Balance_Mismatch')['Exited'].agg(['count', 'mean']))

=== Salary-Balance Mismatch Churn ===


,count,mean
Salary_Balance_Mismatch,,
0,8223,0.216831
1,1777,0.142938


### 5. Retention Strength Assessment & KPIs

In [6]:
# Engagement Retention Ratio: active vs inactive churn comparison
active_churn = df[df['IsActiveMember'] == 1]['Exited'].mean()
inactive_churn = df[df['IsActiveMember'] == 0]['Exited'].mean()
print(f"Engagement Retention Ratio (Inactive / Active Churn): {inactive_churn / active_churn:.2f}x")

# Product Depth Index: ratio of products used vs loyalty
single_prod_churn = df[df['NumOfProducts'] == 1]['Exited'].mean()
multi_prod_churn = df[df['NumOfProducts'] > 1]['Exited'].mean()
print(f"Product Depth Index (Single Prod / Multi Prod Churn): {single_prod_churn / multi_prod_churn:.2f}x")

# High-Balance Disengagement Rate: premium inactive churn
high_bal_inactive_churn = df[(df['Balance'] >= non_zero_balance_median) & (df['IsActiveMember'] == 0)]['Exited'].mean()
print(f"High-Balance Disengagement Rate: {high_bal_inactive_churn:.2%}")

# Credit Card Stickiness Score
cc_churn = df[df['HasCrCard'] == 1]['Exited'].mean()
no_cc_churn = df[df['HasCrCard'] == 0]['Exited'].mean()
print(f"Credit Card Stickiness Score (No CC / CC Churn): {no_cc_churn / cc_churn:.2f}x")

Engagement Retention Ratio (Inactive / Active Churn): 1.88x
Product Depth Index (Single Prod / Multi Prod Churn): 2.17x
High-Balance Disengagement Rate: 31.23%
Credit Card Stickiness Score (No CC / CC Churn): 1.03x


In [7]:
# Save the processed data for Streamlit
df.to_csv('processed_bank_data.csv', index=False)
print("Saved 'processed_bank_data.csv' for the Streamlit dashboard.")

Saved 'processed_bank_data.csv' for the Streamlit dashboard.
